# start_here_rust: Rust/native pySTAMPS guide

This is the beginner-facing Rust notebook.

Read this when you want to understand what the Rust/native backend does, how it stays comparable to MATLAB StaMPS output, and where the speedup is measured. It follows the same teaching pattern as notebook `02_pystamps_stage_execution.ipynb`: first the mental model, then stages, then validation evidence.

The short version:
- Python still orchestrates the pySTAMPS workflow.
- Rust accelerates selected heavy numerical kernels.
- MATLAB StaMPS golden datasets remain the reference for parity.
- This notebook shows focused parity and speed evidence, not a full release audit.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import statistics
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from pystamps.kernels import (
    describe_backend_matrix,
    run_stage4_edge_stats_kernel,
    run_stage7_scla_kernel,
    run_stage8_edge_noise_kernel,
)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'pystamps').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from this notebook')


REPO_ROOT = find_repo_root()
FOCUSED_AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_small_baseline_audit.json'
AUDITED_MANIFEST = REPO_ROOT / 'pystamps' / 'data' / 'audited_workflow_manifest.json'
RUST_SOURCE = REPO_ROOT / 'src' / 'lib.rs'
BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}


def relpath(path: str | Path | None) -> str:
    if path is None:
        return '<missing>'
    candidate = Path(path)
    try:
        return str(candidate.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(candidate)


def modified_utc(path: Path | None) -> str:
    if path is None or not path.exists():
        return '<missing>'
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat(timespec='seconds')


def failure_count(value: object) -> int:
    if isinstance(value, list):
        return len(value)
    if isinstance(value, int):
        return value
    return 0


def run_command(cmd: list[str]) -> tuple[subprocess.CompletedProcess[str], float]:
    t0 = time.perf_counter()
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=BASE_ENV,
        text=True,
        capture_output=True,
        check=False,
    )
    return completed, time.perf_counter() - t0


native_import_error = None
native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
try:
    native_mod = importlib.import_module('pystamps.kernels._stage2_native') if native_spec is not None else None
except Exception as exc:
    native_mod = None
    native_import_error = f'{type(exc).__name__}: {exc}'

native_path = Path(native_spec.origin) if native_spec is not None and native_spec.origin and native_mod is not None else None
artifact_df = pd.DataFrame(
    [
        {'item': 'Python executable', 'value': sys.executable},
        {'item': 'Python version', 'value': sys.version.split()[0]},
        {'item': 'Rust/native module', 'value': relpath(native_path) if native_path else '<not importable>'},
        {'item': 'Rust source', 'value': relpath(RUST_SOURCE)},
        {'item': 'Focused audit artifact', 'value': relpath(FOCUSED_AUDIT_JSON)},
        {'item': 'Audit artifact modified UTC', 'value': modified_utc(FOCUSED_AUDIT_JSON)},
        {'item': 'Native import error', 'value': native_import_error or '<none>'},
    ]
)
display(artifact_df)


,item,value
0,Python executable,/shared/home/rdelprete/PythonProjects/AgenticW...
1,Python version,3.14.2
2,Rust/native module,pystamps/kernels/_stage2_native.cpython-314-x8...
3,Rust source,src/lib.rs
4,Focused audit artifact,inputs_and_outputs/validation_runs/latest_smal...
5,Audit artifact modified UTC,2026-04-22T15:21:37+00:00
6,Native import error,<none>


## Mental model

Keep this picture in mind:

```text
MATLAB StaMPS golden data
  -> pySTAMPS Python workflow
  -> selected Rust/native kernels
  -> pySTAMPS output files
  -> pystamps verify compares output against the golden data
```

Rust is not a second workflow. It is an implementation choice for selected kernels inside the same workflow. The correctness question is therefore not "does Rust produce something plausible?" It is "does the native-backed path still match the trusted StaMPS reference outputs?"


## 1. What native backends are available here?

The table below is the local truth from `describe_backend_matrix()`. The important column is `rust_speed_claim`: only those kernels are used as speed evidence in this notebook.


In [2]:
backend_matrix = describe_backend_matrix()
speed_claim_kernels = {'stage4_edge_stats', 'stage7_scla', 'stage8_edge_noise'}
backend_rows = []
for kernel, info in sorted(backend_matrix.get('kernels', {}).items()):
    supported = info.get('supported_backends', [])
    available = info.get('available_backends', [])
    backend_rows.append(
        {
            'kernel': kernel,
            'python_reference': info.get('baseline_backend', 'python'),
            'native_available_now': 'native' in available,
            'available_backends': ', '.join(available),
            'rust_speed_claim': 'yes' if kernel in speed_claim_kernels else 'no',
            'reader_note': (
                'measured below against Python'
                if kernel in speed_claim_kernels
                else 'available/registered, but not used as speed evidence here'
            ),
        }
    )
backend_df = pd.DataFrame(backend_rows)
display(backend_df)


,kernel,python_reference,native_available_now,available_backends,rust_speed_claim,reader_note
0,stage2_grid_accumulate,python,True,"native, python",no,"available/registered, but not used as speed ev..."
1,stage2_histogram,python,True,"native, python",no,"available/registered, but not used as speed ev..."
2,stage2_topofit,python,True,"native, python",no,"available/registered, but not used as speed ev..."
3,stage2_topofit_coh_row_invariant,python,True,"native, python",no,"available/registered, but not used as speed ev..."
4,stage2_topofit_row_invariant,python,True,"native, python",no,"available/registered, but not used as speed ev..."
5,stage4_edge_stats,python,True,"native, python",yes,measured below against Python
6,stage7_scla,python,True,"cuda, native, python",yes,measured below against Python
7,stage8_edge_noise,python,True,"cuda, native, python",yes,measured below against Python


## 2. How to turn Rust/native execution on

Use the backend controls when you want native execution. The defaults are cautious: Python remains the baseline, and native paths are used only where the registry says they are available.


In [3]:
config_examples = pd.DataFrame(
    [
        {
            'intent': 'Prefer native kernels globally',
            'config': 'runtime:\n  backend: native',
            'when_to_use': 'You want available native kernels for stages such as 4, 7, and 8.',
        },
        {
            'intent': 'Force stage-2 native kernels',
            'config': 'runtime:\n  stage2_kernel_backend: native\n  stage2_native_threads: 8',
            'when_to_use': 'You want stage-2 native registration to be required instead of automatic fallback.',
        },
        {
            'intent': 'Pin one kernel only',
            'config': 'runtime:\n  kernel_backend_overrides:\n    stage7_scla: native',
            'when_to_use': 'You want to test one Rust-backed kernel without changing the whole workflow.',
        },
        {
            'intent': 'Return to the reference path',
            'config': 'runtime:\n  backend: python\n  stage2_kernel_backend: python',
            'when_to_use': 'You are debugging and want the NumPy/Python baseline.',
        },
    ]
)
display(config_examples)


,intent,config,when_to_use
0,Prefer native kernels globally,runtime:\n backend: native,You want available native kernels for stages s...
1,Force stage-2 native kernels,runtime:\n stage2_kernel_backend: native\n s...,You want stage-2 native registration to be req...
2,Pin one kernel only,runtime:\n kernel_backend_overrides:\n sta...,You want to test one Rust-backed kernel withou...
3,Return to the reference path,runtime:\n backend: python\n stage2_kernel_b...,You are debugging and want the NumPy/Python ba...


## 3. Stage-by-stage Rust map

Notebook `02` teaches every pySTAMPS stage. The Rust version is simpler: most stages still run as normal Python orchestration, while stages 2, 4, 7, and 8 have native backend coverage.


In [4]:
stage_map = pd.DataFrame(
    [
        {'stage': '1. Load inputs', 'workflow_goal': 'Read StaMPS-style files into pySTAMPS structures.', 'rust_role': 'none', 'what_to_validate': 'input discovery and file layout'},
        {'stage': '2. Estimate gamma/coherence', 'workflow_goal': 'Estimate coherence-like quantities used for persistent scatterer selection.', 'rust_role': 'registered native kernels for grid/histogram/topofit paths', 'what_to_validate': 'parity first; topofit speed is not claimed here'},
        {'stage': '3. Select persistent scatterers', 'workflow_goal': 'Keep candidate points that pass selection thresholds.', 'rust_role': 'none', 'what_to_validate': 'same selected candidates as reference data'},
        {'stage': '4. Weed adjacent/noisy candidates', 'workflow_goal': 'Use edge statistics to remove weak or redundant candidates.', 'rust_role': 'native `stage4_edge_stats` speed is measured', 'what_to_validate': 'native output matches Python kernel output'},
        {'stage': '5. Merge products', 'workflow_goal': 'Promote patch outputs into merged products.', 'rust_role': 'none', 'what_to_validate': 'merged files match expected structure'},
        {'stage': '6. Unwrap phase', 'workflow_goal': 'Build unwrapped phase products.', 'rust_role': 'none', 'what_to_validate': 'unwrapped products remain comparable'},
        {'stage': '7. Estimate SCLA', 'workflow_goal': 'Estimate spatially correlated look angle error and velocity products.', 'rust_role': 'native `stage7_scla` speed is measured', 'what_to_validate': 'focused StaMPS golden dataset parity'},
        {'stage': '8. Final filtering', 'workflow_goal': 'Estimate and filter spatially correlated noise.', 'rust_role': 'native `stage8_edge_noise` speed is measured', 'what_to_validate': 'native output matches Python kernel output'},
    ]
)
display(stage_map)


,stage,workflow_goal,rust_role,what_to_validate
0,1. Load inputs,Read StaMPS-style files into pySTAMPS structures.,none,input discovery and file layout
1,2. Estimate gamma/coherence,Estimate coherence-like quantities used for pe...,registered native kernels for grid/histogram/t...,parity first; topofit speed is not claimed here
2,3. Select persistent scatterers,Keep candidate points that pass selection thre...,none,same selected candidates as reference data
3,4. Weed adjacent/noisy candidates,Use edge statistics to remove weak or redundan...,native `stage4_edge_stats` speed is measured,native output matches Python kernel output
4,5. Merge products,Promote patch outputs into merged products.,none,merged files match expected structure
5,6. Unwrap phase,Build unwrapped phase products.,none,unwrapped products remain comparable
6,7. Estimate SCLA,Estimate spatially correlated look angle error...,native `stage7_scla` speed is measured,focused StaMPS golden dataset parity
7,8. Final filtering,Estimate and filter spatially correlated noise.,native `stage8_edge_noise` speed is measured,native output matches Python kernel output


## 4. The Rust-relevant stages in plain language

### Stage 2: coherence estimation
Stage 2 is where pySTAMPS estimates quantities used to decide which points are reliable. Native stage-2 kernels are registered, but the current notebook does not use stage-2 topofit as speed evidence because that path preserves parity by falling back to the Python reference when needed.

### Stage 4: edge statistics
Stage 4 compares neighboring candidate points. The Rust/native `stage4_edge_stats` kernel accelerates the repeated edge calculations and is benchmarked below.

### Stage 7: SCLA
Stage 7 estimates spatially correlated look angle error. The focused golden-dataset parity artifact in this notebook is stage-7 small-baseline evidence against MATLAB StaMPS outputs.

### Stage 8: edge noise
Stage 8 uses edge differences again for final space-time filtering. The Rust/native `stage8_edge_noise` kernel is benchmarked below against the Python implementation.


## 5. Parity against MATLAB StaMPS golden data

The next table reads the focused audit artifact. It should show a completed run, no interruption, and passing audit rows. This is intentionally narrow evidence: it proves the focused small-baseline stage-7 parity case, not the full manifest gate.


In [5]:
if not FOCUSED_AUDIT_JSON.exists():
    raise FileNotFoundError(f'Focused audit artifact is missing: {FOCUSED_AUDIT_JSON}')

audit_payload = json.loads(FOCUSED_AUDIT_JSON.read_text(encoding='utf-8'))
audit_rows = audit_payload.get('audits', [])
audit_ok = bool(audit_payload.get('ok') and audit_payload.get('completed') and not audit_payload.get('interrupted'))

audit_status = pd.DataFrame(
    [
        {'check': 'artifact exists', 'result': FOCUSED_AUDIT_JSON.exists()},
        {'check': 'audit completed', 'result': bool(audit_payload.get('completed'))},
        {'check': 'audit interrupted', 'result': bool(audit_payload.get('interrupted'))},
        {'check': 'overall focused audit ok', 'result': audit_ok},
        {'check': 'full manifest claimed here', 'result': False},
    ]
)

audit_df = pd.DataFrame(
    [
        {
            'dataset': audit.get('dataset'),
            'workflow': audit.get('workflow'),
            'status': audit.get('status'),
            'ok': bool(audit.get('ok')),
            'checked_files': int(audit.get('checked', 0)),
            'failed_files': failure_count(audit.get('failed', [])),
            'run_root': relpath(audit.get('run_root')),
            'golden_root': relpath(audit.get('golden_root')),
        }
        for audit in audit_rows
    ]
)

display(audit_status)
display(audit_df)


,check,result
0,artifact exists,True
1,audit completed,True
2,audit interrupted,False
3,overall focused audit ok,True
4,full manifest claimed here,False


,dataset,workflow,status,ok,checked_files,failed_files,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,InSAR_dataset_small_baseline_stage7diag_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...
1,InSAR_dataset_small_baseline_stage7,InSAR_dataset_small_baseline_stage7_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...


## 6. Step-by-step parity checklist

This is the same idea as notebook `02`, but compressed for Rust: identify the trusted StaMPS answer, identify the pySTAMPS run, compare files, and state the limit of the claim.


In [6]:
parity_steps = pd.DataFrame(
    [
        {
            'step': '1. Pick a trusted answer',
            'what_to_check': 'The golden roots are MATLAB StaMPS reference outputs.',
            'evidence': '; '.join(sorted(relpath(audit.get('golden_root')) for audit in audit_rows)),
            'result': 'PASS' if audit_rows else 'FAIL',
        },
        {
            'step': '2. Regenerate pySTAMPS output',
            'what_to_check': 'The audit rows point at fresh validation run roots.',
            'evidence': '; '.join(sorted(relpath(audit.get('run_root')) for audit in audit_rows)),
            'result': 'PASS' if audit_rows else 'FAIL',
        },
        {
            'step': '3. Compare files',
            'what_to_check': 'Every recorded audit row passed with zero failed files.',
            'evidence': f"{int(audit_df['checked_files'].sum()) if not audit_df.empty else 0} checked, {int(audit_df['failed_files'].sum()) if not audit_df.empty else 0} failed",
            'result': 'PASS' if audit_ok else 'FAIL',
        },
        {
            'step': '4. State the limit',
            'what_to_check': 'This notebook proves focused small-baseline stage-7 parity, not the full manifest gate.',
            'evidence': 'Use make audit for full manifest parity.',
            'result': 'PASS',
        },
    ]
)
display(parity_steps)


,step,what_to_check,evidence,result
0,1. Pick a trusted answer,The golden roots are MATLAB StaMPS reference o...,inputs_and_outputs/InSAR_dataset_small_baselin...,PASS
1,2. Regenerate pySTAMPS output,The audit rows point at fresh validation run r...,inputs_and_outputs/validation_runs/20260422_15...,PASS
2,3. Compare files,Every recorded audit row passed with zero fail...,"8 checked, 0 failed",PASS
3,4. State the limit,This notebook proves focused small-baseline st...,Use make audit for full manifest parity.,PASS


## 7. Speed evidence for Rust/native kernels

The benchmark below is synthetic but deterministic. It compares Python and Rust/native implementations for the kernels where this checkout makes a speed claim: stage 4, stage 7, and stage 8.

Read the output table from left to right:
- `parity_ok` must be true, otherwise speed does not matter.
- `rust_faster` must be true.
- `speedup_vs_python` shows how many times faster the native path was in this run.


In [7]:
bench_rows: list[dict[str, object]] = []


def timed(fn: Callable[[], Any], repeats: int) -> tuple[list[float], Any]:
    fn()
    runs: list[float] = []
    output: Any = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        output = fn()
        runs.append(time.perf_counter() - t0)
    return runs, output


def max_abs_array(left: np.ndarray, right: np.ndarray) -> float:
    left_arr = np.asarray(left)
    right_arr = np.asarray(right)
    if np.array_equal(left_arr, right_arr, equal_nan=True):
        return 0.0
    finite = np.isfinite(left_arr) & np.isfinite(right_arr)
    if not np.all(np.equal(left_arr[~finite], right_arr[~finite])):
        return float('inf')
    return float(np.max(np.abs(left_arr[finite] - right_arr[finite]))) if np.any(finite) else 0.0


def max_abs_output(left: dict[str, np.ndarray], right: dict[str, np.ndarray]) -> float:
    if set(left) != set(right):
        return float('inf')
    return max(max_abs_array(np.asarray(left[key]), np.asarray(right[key])) for key in sorted(left))


def record_kernel(
    kernel: str,
    python_fn: Callable[[], dict[str, np.ndarray]],
    native_fn: Callable[[], dict[str, np.ndarray]],
    tolerance: float,
    repeats: int,
) -> None:
    python_runs, python_output = timed(python_fn, repeats)
    native_runs, native_output = timed(native_fn, repeats)
    python_mean = statistics.mean(python_runs)
    native_mean = statistics.mean(native_runs)
    max_abs = max_abs_output(python_output, native_output)
    bench_rows.append(
        {
            'kernel': kernel,
            'python_mean_sec': python_mean,
            'native_mean_sec': native_mean,
            'speedup_vs_python': python_mean / native_mean if native_mean > 0 else float('inf'),
            'max_abs': max_abs,
            'tolerance': tolerance,
            'parity_ok': bool(max_abs <= tolerance),
            'rust_faster': bool(native_mean < python_mean),
            'python_runs_sec': ', '.join(f'{value:.4f}' for value in python_runs),
            'native_runs_sec': ', '.join(f'{value:.4f}' for value in native_runs),
        }
    )


if native_mod is None:
    bench_rows.append(
        {
            'kernel': '<native unavailable>',
            'parity_ok': False,
            'rust_faster': False,
            'reason': 'build/install the Rust extension before measuring native speed',
        }
    )
else:
    rng = np.random.default_rng(321)
    repeats = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '1'))

    n_ps4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_PS', '30000'))
    n_ifg4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_IFG', '32'))
    n_edge4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_EDGES', '60000'))
    ph_weed = np.exp(1j * rng.normal(size=(n_ps4, n_ifg4))).astype(np.complex128)
    node_a4 = rng.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    node_b4 = rng.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    bperp4 = np.linspace(-80.0, 120.0, n_ifg4, dtype=np.float64)
    day4 = np.linspace(0.0, 365.0, n_ifg4, dtype=np.float64)
    record_kernel(
        'stage4_edge_stats',
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='python'),
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='native'),
        1.0e-10,
        repeats,
    )

    n_ps7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_PS', '30000'))
    n_ifg7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_IFG', '36'))
    ph_proc = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    ph_mean_v = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    bperp_mat = rng.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    unwrap_ix = np.arange(n_ifg7, dtype=np.int64)
    solve_ix = np.arange(n_ifg7, dtype=np.int64)
    day7 = np.linspace(0.0, 365.0, n_ifg7, dtype=np.float64)
    ifg_std = np.full(n_ifg7, 10.0, dtype=np.float64)
    record_kernel(
        'stage7_scla',
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='python'),
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='native'),
        1.0e-4,
        repeats,
    )

    n_ps8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_PS', '50000'))
    n_ifg8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_IFG', '36'))
    n_edge8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_EDGES', '120000'))
    uw_ph = np.exp(1j * rng.normal(size=(n_ps8, n_ifg8))).astype(np.complex64)
    node_a8 = rng.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    node_b8 = rng.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    record_kernel(
        'stage8_edge_noise',
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='python'),
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='native'),
        1.0e-5,
        repeats,
    )

bench_df = pd.DataFrame(bench_rows)
if not bench_df.empty and {'speedup_vs_python', 'parity_ok', 'rust_faster'} <= set(bench_df.columns):
    speed_ok = bool(bench_df['parity_ok'].all() and bench_df['rust_faster'].all())
    min_speedup = round(float(bench_df['speedup_vs_python'].min()), 3)
else:
    speed_ok = False
    min_speedup = '<not measured>'

bench_display = bench_df.copy()
for column in ['python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance']:
    if column in bench_display:
        bench_display[column] = bench_display[column].map(lambda value: round(float(value), 6) if pd.notna(value) else value)

speed_summary = pd.DataFrame(
    [
        {'check': 'all measured kernels match Python numerically', 'result': bool(bench_df.get('parity_ok', pd.Series(dtype=bool)).all())},
        {'check': 'all measured Rust/native kernels are faster', 'result': bool(bench_df.get('rust_faster', pd.Series(dtype=bool)).all())},
        {'check': 'minimum native speedup versus Python', 'result': min_speedup},
        {'check': 'stage-2 topofit speed claimed', 'result': False},
    ]
)

display(bench_display.reindex(columns=['kernel', 'python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance', 'parity_ok', 'rust_faster', 'python_runs_sec', 'native_runs_sec', 'reason']).dropna(axis=1, how='all'))
display(speed_summary)


,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster,python_runs_sec,native_runs_sec
0,stage4_edge_stats,4.987969,0.665619,7.493725,0.000000,0.00000,True,True,4.9880,0.6656
1,stage7_scla,0.074711,0.012870,5.805305,0.000031,0.00010,True,True,0.0747,0.0129
2,stage8_edge_noise,0.096567,0.030496,3.166512,0.000000,0.00001,True,True,0.0966,0.0305


,check,result
0,all measured kernels match Python numerically,True
1,all measured Rust/native kernels are faster,True
2,minimum native speedup versus Python,3.167
3,stage-2 topofit speed claimed,False


## 8. Final reading checklist

If the next table is all green, this notebook has shown the local native module, a focused StaMPS parity artifact, a step-by-step parity explanation, and live Rust speed measurements.


In [8]:
reader_checks = pd.DataFrame(
    [
        {'question': 'Is the Rust/native module importable?', 'answer': native_mod is not None},
        {'question': 'Did the focused audit complete without interruption?', 'answer': audit_ok},
        {'question': 'Did the notebook show the golden-data parity steps?', 'answer': bool(not parity_steps.empty and (parity_steps['result'] == 'PASS').all())},
        {'question': 'Are the measured Rust/native kernels faster than Python?', 'answer': speed_ok},
        {'question': 'Did this notebook claim full manifest parity?', 'answer': False},
        {'question': 'Did this notebook claim stage-2 topofit speed?', 'answer': False},
    ]
)
display(reader_checks)


,question,answer
0,Is the Rust/native module importable?,True
1,Did the focused audit complete without interru...,True
2,Did the notebook show the golden-data parity s...,True
3,Are the measured Rust/native kernels faster th...,True
4,Did this notebook claim full manifest parity?,False
5,Did this notebook claim stage-2 topofit speed?,False


## What to open next

- Open `02_pystamps_stage_execution.ipynb` for the full Python workflow stage walk-through.
- Open `03_pystamps_verification.ipynb` to learn the verification and audit tools.
- Open `04_rust_end2end_parity_validation.ipynb` for the evidence appendix that reruns `pystamps verify` and the native kernel smoke tests.
- Run `make audit` when you need the full maintained parity gate instead of this focused notebook view.
